Представим себе, что мы хотим подготовить корпус текстов для морфосинтаксической разметки и дальнейшего поиска по ним. Нам необходимо:

Подобрать данные - наш корпус игрушечный, но вам необходимо выбрать столько текстов, чтобы суммарно в них было не меньше 10 тысяч слов. Объясните свой выбор текстов: если не будет объяснения, минус балл. Можете пользоваться wikipedia-api или даже поискать готовые датасеты текстов (язык не имеет большого значения, можно взять что-то кроме русского).

Удостовериться, что тексты чистые, в них нет ссылок, хештегов, мусора, оставшегося после html-обвязки, и подобного. Если что-то из этого есть, почистить с помощью регулярных выражений.

Посчитать статистику для нашего корпуса и красиво ее представить (с помощью f-строк): сколько в корпусе документов, сколько в каждом документе слов, сколько предложений.

Вывести распределение предложений по длинам: сколько в корпусе в процентах предложений длиной до 10 слов, от 11 до 20 слов и так далее, а также сколько слов суммарно приходится на предложения каждой категории длины. Вывод должен выглядеть примерно так:

 1-10 слов:
 1234 слова всего
 4% (12 предложений)
 11-20 слов:
 ...


In [ ]:
!pip3 install wikipedia-api

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia-api: filename=Wikipedia_API-0.7.1-py3-none-any.whl size=14346 sha256=6b574cc3affa002058ea5b8cb125ce4699c2b90b845549a97ceb96c39b88ba3e
  Stored in directory: /root/.cache/pip/wheels/4c/96/18/b9201cc3e8b47b02b510460210cfd832ccf10c0c4dd0522962
Successfully built wikipedia-api


In [ ]:
import wikipediaapi
import re

In [ ]:


# Функция для получения текстов из Википедии
def fetch_wikipedia_texts(page_titles):
    wiki_wiki = wikipediaapi.Wikipedia(
        user_agent = 'phtchk', language='en', extract_format=wikipediaapi.ExtractFormat.WIKI
)
    texts = []

    for title in page_titles:
        page = wiki_wiki.page(title)
        if page.exists():
            texts.append(page.text)

    return texts

# Функция для очистки текста
def clean_text(text):
    # Удаляем ссылки, хештеги и специальный HTML
    text = re.sub(r'\[.*?\]', '', text)  # Удаляем ссылки и аннотации
    text = re.sub(r'\#\w+', '', text)    # Удаляем хештеги
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Удаляем URL
    text = re.sub(r'[^A-Za-z0-9\s.,!?\'-]', '', text)  # Оставляем только буквы и знаки препинания
    text = text.strip()
    return text

# Подбираем страницы для создания корпуса
page_titles = ["Python (programming language)", "Machine learning", "Artificial intelligence",
               "Data science", "Deep learning", "Natural language processing"]

# Получаем текстов
texts = fetch_wikipedia_texts(page_titles)

# Чистим тексты
cleaned_texts = []
for text in texts:
  cleaned_texts.append(clean_text(text))

total_words = 0
for text in cleaned_texts:
  total_words += len(text.split())

print(f"Собрано текстов: {len(cleaned_texts)}, Общее количество слов: {total_words}")

#делаем пустой словарик для записи результатов - i номер категории, каунт и вордс ну понятно
length_distribution = {}
for i in range(0, 40):
  length_distribution.update({i: {'sentences': 0, 'words': 0}})
# тут мы по сути ограничили количество категорий до 40 (мы максимум дойдём до 400 слов, более длинных предложений в тексте точно нету)

# Подсчет длины предложений и слов
total_sentences = 0
for text in cleaned_texts:
    sentences = re.split(r'[.!?]', text)  # Разделяем на предложения
    total_sentences += len(sentences)
    for sentence in sentences:
        words = sentence.split()
        word_count = len(words)
        if word_count > 0: #почему-то есть пустые предложения, они нам не нужны совсем
          # Определяем категорию
          category = (word_count - 1) // 10  # 1-10 будет 0, 11-20 будет 1 и т.д.
          length_distribution[category]['sentences'] += 1
          length_distribution[category]['words'] += word_count

print("\nРаспределение предложений по длинам:")

for category in range(0, 40):  # пробегаемся по категориям от 0 до 40, то есть до 400 слов
    sentences_in_cat = length_distribution[category]['sentences']
    words_in_cat = length_distribution[category]['words']
    percentage = (sentences_in_cat / total_sentences * 100)
    print(f"{(category) * 10 + 1}-{(category + 1)* 10} слов: {words_in_cat} слов всего, {percentage:.2f}% ({sentences_in_cat} предложений)")



Собрано текстов: 6, Общее количество слов: 42708

Распределение предложений по длинам:
1-10 слов: 3108 слов всего, 26.94% (643 предложений)
11-20 слов: 13063 слов всего, 35.36% (844 предложений)
21-30 слов: 14179 слов всего, 23.96% (572 предложений)
31-40 слов: 6770 слов всего, 8.21% (196 предложений)
41-50 слов: 2961 слов всего, 2.76% (66 предложений)
51-60 слов: 942 слов всего, 0.71% (17 предложений)
61-70 слов: 570 слов всего, 0.38% (9 предложений)
71-80 слов: 158 слов всего, 0.08% (2 предложений)
81-90 слов: 167 слов всего, 0.08% (2 предложений)
91-100 слов: 197 слов всего, 0.08% (2 предложений)
101-110 слов: 101 слов всего, 0.04% (1 предложений)
111-120 слов: 0 слов всего, 0.00% (0 предложений)
121-130 слов: 0 слов всего, 0.00% (0 предложений)
131-140 слов: 0 слов всего, 0.00% (0 предложений)
141-150 слов: 0 слов всего, 0.00% (0 предложений)
151-160 слов: 0 слов всего, 0.00% (0 предложений)
161-170 слов: 163 слов всего, 0.04% (1 предложений)
171-180 слов: 174 слов всего, 0.04% (1 

In [ ]:
'''for key in length_distribution.keys():
  sentences = length_distribution[key]['sentences']
  total_in_category = length_distribution[key]['words']
  percentage = (sentences / total_sentences * 100)
  print(f"{(key) * 10 + 1}-{(key + 1)* 10} слов: {total_in_category} слов всего, {percentage:.2f}% ({sentences} предложений)")
'''

1-10 слов: 3108 слов всего, 26.94% (643 предложений)
11-20 слов: 13063 слов всего, 35.36% (844 предложений)
21-30 слов: 14179 слов всего, 23.96% (572 предложений)
31-40 слов: 6770 слов всего, 8.21% (196 предложений)
41-50 слов: 2961 слов всего, 2.76% (66 предложений)
51-60 слов: 942 слов всего, 0.71% (17 предложений)
61-70 слов: 570 слов всего, 0.38% (9 предложений)
71-80 слов: 158 слов всего, 0.08% (2 предложений)
81-90 слов: 167 слов всего, 0.08% (2 предложений)
91-100 слов: 197 слов всего, 0.08% (2 предложений)
101-110 слов: 101 слов всего, 0.04% (1 предложений)
111-120 слов: 0 слов всего, 0.00% (0 предложений)
121-130 слов: 0 слов всего, 0.00% (0 предложений)
131-140 слов: 0 слов всего, 0.00% (0 предложений)
141-150 слов: 0 слов всего, 0.00% (0 предложений)
151-160 слов: 0 слов всего, 0.00% (0 предложений)
161-170 слов: 163 слов всего, 0.04% (1 предложений)
171-180 слов: 174 слов всего, 0.04% (1 предложений)
181-190 слов: 0 слов всего, 0.00% (0 предложений)
191-200 слов: 197 слов в